# TP 3 — Partie 3 : mesurer, puis optimiser

**Règle d'or :** on n'optimise que ce que l'on a **mesuré**. Les vues *Gantt* d'Airflow et l'interface Spark (http://localhost:4040) montrent où passe le temps ; les tables de métadonnées d'Iceberg montrent comment les données sont rangées.

Trois leviers classiques dans un Lakehouse :
1. **le traitement incrémental** : ne traiter que ce qui a changé (déjà fait en partie 2) ;
2. **le rangement des fichiers** : éviter les milliers de petits fichiers, et ne garder que les versions utiles ;
3. **le partitionnement** : ranger les données pour que les requêtes ne lisent que ce dont elles ont besoin.

> **⚠️ Un seul processus Spark à la fois.** Arrêtez Airflow (`docker compose --profile orchestration stop airflow`) ou attendez la fin de son exécution.

In [ ]:
import time
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("tp3-optimiser").getOrCreate()


def a_completer(quoi):
    """Signale clairement une partie du TP qui n'a pas encore été écrite."""
    raise NotImplementedError(f"⚠️ Cellule à compléter : {quoi}")


def chrono(requete):
    """Exécute une requête Spark SQL et renvoie sa durée en secondes."""
    debut = time.time()
    spark.sql(requete).collect()
    return round(time.time() - debut, 2)


def fichiers(table):
    """Nombre et taille des fichiers de données d'une table Iceberg."""
    return spark.sql(f"""
        SELECT COUNT(*) AS nb_fichiers,
               ROUND(SUM(file_size_in_bytes) / 1e6, 1) AS taille_mo,
               ROUND(AVG(file_size_in_bytes) / 1e3, 0) AS taille_moyenne_ko
        FROM lakehouse.{table}.files""")

## Étape 1 — Le problème des petits fichiers

Chaque écriture (chaque exécution du pipeline, chaque `append`) ajoute de **nouveaux fichiers** et une nouvelle **version** (*snapshot*) de la table. Après quelques semaines d'exécutions quotidiennes, une table peut compter des milliers de petits fichiers : chaque requête doit tous les ouvrir.

In [ ]:
for t in ["bronze.photos", "silver.photos", "gold.photos_vecteurs", "silver.avis"]:
    print(t)
    fichiers(t).show()
spark.sql("SELECT committed_at, operation, summary['added-data-files'] AS fichiers_ajoutes "
          "FROM lakehouse.silver.photos.snapshots ORDER BY committed_at").show(truncate=False)

Simulons trois mois d'exécutions : 90 petites écritures d'une seule ligne dans une copie de `silver.photos`.

In [ ]:
spark.sql("CREATE OR REPLACE TABLE lakehouse.silver.photos_demo USING iceberg AS SELECT * FROM lakehouse.silver.photos")
for i in range(90):
    spark.sql(f"INSERT INTO lakehouse.silver.photos_demo SELECT * FROM lakehouse.silver.photos LIMIT 1 OFFSET {i}")
fichiers("silver.photos_demo").show()
avant = chrono("SELECT vendeur, COUNT(*) FROM lakehouse.silver.photos_demo GROUP BY vendeur")
print("durée de la requête :", avant, "s")

## Étape 2 — Compacter et nettoyer

Iceberg fournit des **procédures de maintenance**, appelées avec `CALL` :
- `rewrite_data_files` regroupe les petits fichiers en gros fichiers (compactage) ;
- `expire_snapshots` supprime les anciennes versions, et les fichiers qu'elles seules utilisaient.

Complétez les deux appels sur `silver.photos_demo`, puis mesurez de nouveau.

In [ ]:
a_completer("CALL lakehouse.system.rewrite_data_files(...) puis expire_snapshots(...)")  # À COMPLÉTER
fichiers("silver.photos_demo").show()
apres = chrono("SELECT vendeur, COUNT(*) FROM lakehouse.silver.photos_demo GROUP BY vendeur")
print(f"durée : {avant} s → {apres} s")

> **❓ Question 1.** Que perd-on en expirant les anciennes versions ? Pourquoi garde-t-on en général quelques jours d'historique plutôt qu'une seule version ?

## Étape 3 — Partitionner

Une table **partitionnée** range ses fichiers par valeur d'une colonne : ici, un dossier par vendeur. Une requête qui filtre sur le vendeur ne lit alors que ses fichiers (*partition pruning*).

Créez `gold.photos_vecteurs_part`, copie de `gold.photos_vecteurs` enrichie du vendeur (jointure avec `silver.photos`), **partitionnée par `vendeur`** : `CREATE TABLE ... USING iceberg PARTITIONED BY (vendeur) AS SELECT ...`.

In [ ]:
a_completer("CREATE TABLE lakehouse.gold.photos_vecteurs_part ... PARTITIONED BY (vendeur)")  # À COMPLÉTER
spark.sql("""CREATE OR REPLACE TABLE lakehouse.gold.photos_vecteurs_plat USING iceberg AS
             SELECT * FROM lakehouse.gold.photos_vecteurs_part""")
# Iceberg garde pour chaque fichier la plus petite et la plus grande valeur de chaque colonne :
# un fichier dont l'intervalle [min, max] ne contient pas '03' n'est même pas ouvert.
for t in ["gold.photos_vecteurs_plat", "gold.photos_vecteurs_part"]:
    stats = spark.sql(f"""
        SELECT COUNT(*) AS total,
               SUM(CASE WHEN readable_metrics.vendeur.lower_bound <= '03'
                         AND readable_metrics.vendeur.upper_bound >= '03' THEN 1 ELSE 0 END) AS a_lire
        FROM lakehouse.{t}.files""").first()
    lignes = spark.sql(f"SELECT * FROM lakehouse.{t} WHERE vendeur = '03'").count()
    print(f"{t:28s} fichiers : {stats.total:3d} | à lire pour le vendeur 03 : {stats.a_lire:3d} | lignes : {lignes}")
spark.sql("SELECT partition, record_count, file_count FROM lakehouse.gold.photos_vecteurs_part.partitions").show()

Regardez le plan d'exécution de la requête filtrée sur la table partitionnée : la ligne `BatchScan` indique le filtre (`filters=vendeur = '03'`) transmis à Iceberg, qui choisit les fichiers à lire avant même que Spark ne les ouvre.

*Remarque :* avec un simple `COUNT(*)`, Spark ne lirait aucun fichier : il répondrait avec les statistiques de la table (`LocalTableScan`). On sélectionne donc des colonnes.

In [ ]:
spark.sql("SELECT chemin, vendeur FROM lakehouse.gold.photos_vecteurs_part WHERE vendeur = '03'").explain()

> **❓ Question 2.** Partitionner par vendeur est-il toujours un bon choix ? Que se passerait-il avec une partition par `item_id` (une partition par produit) ? Quelle colonne choisiriez-vous pour `silver.avis` ?

## Étape 4 — Automatiser la maintenance

La maintenance n'est utile que si elle est **régulière** : c'est une tâche pour Airflow. Complétez le DAG `dags/maintenance_iceberg.py` : chaque semaine, il compacte les tables du TP 3 et ne garde que 7 jours d'historique (script `pipeline/maintenance.py`, fourni).

In [ ]:
!cp -n dag_maintenance_iceberg.py dags/maintenance_iceberg.py && cat dags/maintenance_iceberg.py

> **❓ Question 3.** Le DAG de maintenance et le DAG `catalogue_photos` écrivent dans les mêmes tables. Que risque-t-on s'ils tournent en même temps ? Comment Iceberg et Airflow limitent-ils ce risque ?

In [ ]:
spark.sql("DROP TABLE IF EXISTS lakehouse.silver.photos_demo PURGE")
spark.sql("DROP TABLE IF EXISTS lakehouse.gold.photos_vecteurs_plat PURGE")
print("tables de démonstration supprimées")